# OLMoE Training on Kaggle

**Before running — checklist:**
- Kaggle > Settings > Accelerator: `GPU T4 x2` or `P100`
- Kaggle > Settings > Internet: **ON**
- Kaggle > Add-ons > Secrets: add secret named `WANDB_API_KEY`
- Kaggle > Input: attach your tokenized dataset (contains `.npy` files)
- Edit Cell 2 to set the correct glob path to your `.npy` files

**Run cells in order. Do not skip any cell.**

In [ ]:
# ── Cell 1: System check ──────────────────────────────────────────────────────
import subprocess, os, sys, torch, shutil

if shutil.which("nvidia-smi"):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
else:
    raise EnvironmentError(
        "\n'nvidia-smi' not found — no GPU is attached.\n"
        "Fix: Kaggle notebook > Settings (right panel) > Accelerator > GPU T4 x2 or P100 > Save\n"
        "The kernel will restart. Then re-run this cell."
    )

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
N_GPU = torch.cuda.device_count()
print(f"GPUs    : {N_GPU}")
for i in range(N_GPU):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

assert N_GPU > 0, "No GPU detected. Enable a GPU accelerator in Kaggle settings."

df = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout
print(df)
free_gb = float(subprocess.run(
    ['df', '--output=avail', '-BG', '/kaggle/working'],
    capture_output=True, text=True
).stdout.strip().split()[-1].replace('G',''))
if free_gb < 8:
    print(f"WARNING: only {free_gb:.1f} GB free — may be tight.")
else:
    print(f"Disk OK: {free_gb:.1f} GB free.")

In [ ]:
# ── Cell 2: Configuration — EDIT THIS ─────────────────────────────────────────
import glob

# Set this to the path of your tokenized .npy files inside /kaggle/input/
TOKENIZED_DATA_PATHS = sorted(glob.glob("/kaggle/input/datasets/tristanmartin22/olmoe-dataset/*.npy"))

assert len(TOKENIZED_DATA_PATHS) > 0, (
    "\nNo .npy files found.\n"
    "Possible fixes:\n"
    "  1. Check the dataset is attached in Kaggle > Input\n"
    "  2. Run: !find /kaggle/input -name '*.npy' to see actual paths\n"
    "  3. Update the glob pattern above."
)

print(f"Found {len(TOKENIZED_DATA_PATHS)} tokenized shard(s):")
for p in TOKENIZED_DATA_PATHS:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {p}  ({size_mb:.0f} MB)")

# ── Batch size config ──────────────────────────────────────────────────────────
# global_train_batch_size: total tokens per optimizer step across all GPUs.
#   Must be divisible by N_GPU. Larger = better gradient estimates but more
#   memory per gradient-accumulation microbatch (only if microbatch > 1).
# device_train_microbatch_size: sequences processed per GPU per forward pass.
#   Peak GPU memory scales with this value. Keep at 1 if OOMing.
#   grad_accum_steps = (global_train_batch_size // N_GPU) // device_train_microbatch_size
GLOBAL_TRAIN_BATCH_SIZE    = 16   # total batch across all GPUs
DEVICE_TRAIN_MICROBATCH_SIZE = 1  # per-GPU per-step (controls peak VRAM)


In [ ]:
# ── Cell 3: W&B secret ────────────────────────────────────────────────────────
# Requires WANDB_API_KEY added in Kaggle > Add-ons > Secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_ENTITY"]  = "iliass-lasri-team"
os.environ["WANDB_PROJECT"] = "olmoe-1"
os.environ["WANDB_MODE"]    = "online"
print("W&B configured.")

In [ ]:
# ── Cell 4: Clone repo ────────────────────────────────────────────────────────
REPO_URL         = "https://github.com/iliasslasri/OLMoE.git"
OLMOE_DIR        = "/kaggle/working/OLMoE"
SUBMODULE_COMMIT = "04a2da53db172bd9a0450705592ed50888bdcaa7"

if not os.path.exists(OLMOE_DIR):
    # Shallow clone main repo (--depth 1 saves ~400 MB)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, OLMOE_DIR], check=True)

    # Override submodule URL from SSH (git@github.com:...) to HTTPS.
    # Kaggle has no SSH keys, so the default URL in .gitmodules fails with
    # "Host key verification failed".
    subprocess.run(
        ["git", "config", "submodule.OLMo.url", "https://github.com/allenai/OLMo.git"],
        cwd=OLMOE_DIR, check=True
    )

    # Init submodule WITHOUT --depth so the pinned commit is always reachable.
    # --depth 1 on submodules can fail when the pinned commit is not the latest.
    subprocess.run(["git", "submodule", "update", "--init"], cwd=OLMOE_DIR, check=True)

    # Hard-pin to the exact commit the main repo expects
    subprocess.run(["git", "checkout", SUBMODULE_COMMIT],
                   cwd=f"{OLMOE_DIR}/OLMo", check=True)
else:
    print("Repo already present, skipping clone.")

os.chdir(OLMOE_DIR)
print("cwd:", os.getcwd())
print("main:", subprocess.run(["git", "log", "--oneline", "-1"],
                               capture_output=True, text=True).stdout.strip())
print("OLMo:", subprocess.run(["git", "log", "--oneline", "-1"],
                               capture_output=True, text=True, cwd="OLMo").stdout.strip())

In [ ]:
# ── Cell 5: Install dependencies ──────────────────────────────────────────────
#
# ROOT CAUSE of all Vast.ai / Kaggle crashes:
#   ai2-olmo-core==0.1.0 pins huggingface_hub to ~0.36.x.
#   transformers (>=4.40) requires huggingface_hub>=1.3.0.
#   dolma depends on transformers → import fails.
#
# KAGGLE sys.path / ABI GOTCHAS:
#   1. Old huggingface_hub at /usr/local/lib/python3.12/dist-packages/ shadows
#      pip upgrades. Fix: install to OVERRIDE_DIR and prepend to sys.path.
#      Use --no-deps to avoid pulling in packages with a different compiled ABI.
#   2. `pip install -e` creates a .pth only processed at interpreter startup.
#      Fix: manually add OLMo source dir to sys.path.
#   3. OLMo[train] requires numpy<2.0 (hard dep via ai2-olmo-core), so numpy
#      will downgrade from 2.x → 1.x. This breaks Kaggle's pre-compiled
#      pyarrow/pandas in the KERNEL (ABI mismatch). Do NOT import olmo/dolma
#      in the kernel — the training SUBPROCESS is unaffected because it starts
#      fresh and never imports kaggle_gcp/pyarrow.
#   4. OLMo[train] may downgrade torch (its deps conflict with Kaggle's 2.9.x),
#      making the system torchvision ABI-incompatible. We cannot pin torch to
#      prevent downgrade (resolution fails). Fix: let OLMo pick its torch, then
#      detect the installed version and reinstall torchvision from PyTorch's
#      wheel index for that exact torch — guarantees ABI compatibility.
#   5. megablocks (required for moe_dropless + sparse MoE) must be compiled
#      from source AFTER torch is downgraded by OLMo[train]. Use
#      --force-reinstall so pip recompiles against the new torch ABI rather
#      than reusing a stale .so built for torch 2.9.x. megablocks cannot be
#      verified in the kernel (kernel torch is still 2.9 in memory); the
#      training subprocess starts fresh and loads correctly.
#   6. dolma is NOT imported in the notebook kernel — data is pre-tokenized
#      locally. Only the training subprocess needs olmo (via PYTHONPATH).

import sys

OVERRIDE_DIR = "/kaggle/working/_pip_overrides"
OLMO_SRC     = f"{OLMOE_DIR}/OLMo"

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        raise RuntimeError(f"pip failed: {' '.join(args)}")

def pip_override(*args):
    """Install ONLY the named packages (--no-deps) into OVERRIDE_DIR.
    --no-deps prevents pulling in packages with a different compiled ABI."""
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", OVERRIDE_DIR, "--no-deps"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        raise RuntimeError(f"pip_override failed: {' '.join(args)}")

def purge_modules(*prefixes):
    stale = [k for k in sys.modules
             if any(k == p or k.startswith(p + ".") for p in prefixes)]
    for k in stale:
        del sys.modules[k]
    if stale:
        print(f"  Purged {len(stale)} cached module(s): {prefixes}")

os.makedirs(OVERRIDE_DIR, exist_ok=True)

# ── Step 1: OLMo with all training extras ────────────────────────────────────
print("[1/5] Installing OLMo[train]...")
pip("-e", "OLMo[train]")

if OLMO_SRC not in sys.path:
    sys.path.insert(1, OLMO_SRC)
    print(f"  Added {OLMO_SRC} to sys.path")

# ── Step 2: megablocks — must come AFTER OLMo[train] so it compiles against
# the downgraded torch. --force-reinstall ensures recompilation even if a
# stale wheel (built for torch 2.9.x) is already present on disk.
print("[2/5] Installing megablocks (OLMoE fork, force-recompile)...")
pip("--force-reinstall", "git+https://github.com/Muennighoff/megablocks.git@olmoe")

# ── Step 3: Reinstall torchvision matching the (possibly downgraded) torch ────
print("[3/5] Detecting torch version after OLMo install...")
detect = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True
)
new_torch_full = detect.stdout.strip()
cuda_tag = new_torch_full.split("+")[1] if "+" in new_torch_full else "cpu"
print(f"  Detected torch=={new_torch_full}, CUDA tag: {cuda_tag}")

print("[4/5] Reinstalling torchvision for detected torch...")
pip("--force-reinstall", "torchvision",
    "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}")
purge_modules("torchvision")

# ── Step 4: Force-upgrade huggingface_hub ────────────────────────────────────
print("[5/5] Fixing huggingface_hub / tokenizers versions...")
pip_override("huggingface_hub>=1.3.0,<2.0")
pip_override("tokenizers>=0.22.0,<=0.23.0")

if OVERRIDE_DIR not in sys.path:
    sys.path.insert(0, OVERRIDE_DIR)
    print(f"  Inserted {OVERRIDE_DIR} at sys.path[0]")

purge_modules("huggingface_hub", "tokenizers")

# ── Verify ────────────────────────────────────────────────────────────────────
# NOTE: megablocks is NOT verified here. Its C extension (megablocks_ops.so)
# is compiled against torch 2.3.x but the kernel still has torch 2.9 loaded
# in memory — importing it here would crash. The training subprocess starts
# fresh (no cached modules) and will load megablocks correctly.
print("Verifying imports...")
try:
    from huggingface_hub import is_offline_mode  # noqa
    print("  ✓ huggingface_hub version OK")
except ImportError as e:
    raise RuntimeError(f"huggingface_hub fix failed: {e}\nsys.path[0]={sys.path[0]}")

import importlib
for mod in ["omegaconf", "wandb"]:
    importlib.import_module(mod)
    print(f"  ✓ {mod}")

import huggingface_hub, tokenizers as hf_tok
print(f"  huggingface_hub=={huggingface_hub.__version__}")
print(f"  tokenizers=={hf_tok.__version__}")
print(f"  torch (subprocess)=={new_torch_full}")
print("All dependencies OK. megablocks + olmo verified by Cell 8 dry-run subprocess.")


In [ ]:
# ── Cell 6: Download tokenizer file ───────────────────────────────────────────
# The config references `tokenizers/allenai_gpt-neox-olmo-dolma-v1_5.json`
# as a LOCAL file path relative to the working directory.
# We download it directly with wget (avoids any huggingface_hub version issues).

os.makedirs("tokenizers", exist_ok=True)
TOK_PATH = "tokenizers/allenai_gpt-neox-olmo-dolma-v1_5.json"

if not os.path.exists(TOK_PATH):
    print("Downloading tokenizer...")
    result = subprocess.run([
        "wget", "-q",
        "https://huggingface.co/allenai/gpt-neox-olmo-dolma-v1_5/resolve/main/tokenizer.json",
        "-O", TOK_PATH
    ], capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(TOK_PATH):
        raise RuntimeError(
            "Tokenizer download failed. Check internet is enabled "
            "(Kaggle > Settings > Internet ON).\n" + result.stderr
        )
    size_kb = os.path.getsize(TOK_PATH) / 1024
    print(f"  Saved to {TOK_PATH} ({size_kb:.0f} KB)")
    assert size_kb > 100, f"File too small ({size_kb:.0f} KB) — download may have failed silently."
else:
    print(f"Tokenizer already present at {TOK_PATH}")

In [ ]:
# ── Cell 7: Patch training config ─────────────────────────────────────────────
import re, pathlib, yaml

# ── Pre-flight checks (fail before touching the file) ─────────────────────────
if 'TOKENIZED_DATA_PATHS' not in dir() or len(TOKENIZED_DATA_PATHS) == 0:
    raise RuntimeError("TOKENIZED_DATA_PATHS is empty or undefined — run Cell 2 first!")
if 'N_GPU' not in dir() or N_GPU == 0:
    raise RuntimeError("N_GPU undefined — run Cell 1 first!")
if 'OLMOE_DIR' not in dir():
    raise RuntimeError("OLMOE_DIR undefined — run Cell 4 first!")
if GLOBAL_TRAIN_BATCH_SIZE % N_GPU != 0:
    raise ValueError(f"global_train_batch_size ({GLOBAL_TRAIN_BATCH_SIZE}) must be divisible by N_GPU ({N_GPU})")
device_batch = GLOBAL_TRAIN_BATCH_SIZE // N_GPU
if device_batch % DEVICE_TRAIN_MICROBATCH_SIZE != 0:
    raise ValueError(f"device_batch ({device_batch}) must be divisible by microbatch ({DEVICE_TRAIN_MICROBATCH_SIZE})")
grad_accum = device_batch // DEVICE_TRAIN_MICROBATCH_SIZE
print(f"Batch: global={GLOBAL_TRAIN_BATCH_SIZE}, per-GPU={device_batch}, microbatch={DEVICE_TRAIN_MICROBATCH_SIZE}, grad_accum={grad_accum}")

CONFIG_PATH = pathlib.Path(OLMOE_DIR) / "configs/olmoe-small.yml"
if not CONFIG_PATH.exists():
    raise RuntimeError(f"Config not found: {CONFIG_PATH} — run Cell 4 first!")

# ── Activation checkpointing strategy ─────────────────────────────────────────
# MoE models (block_type: moe) only support TWO strategies (train.py:136):
#   fine_grained  → checkpoints individual ops (attn+norms) within every block,
#                   but explicitly SKIPS the MoE FFN (model.py:822). This is the
#                   only checkpointing mode that reduces attn peak memory for MoE.
#   null          → no checkpointing. 0% overhead but highest memory usage.
#
# one_in_two / one_in_four / whole_layer are DENSE-model strategies and will
# raise OLMoConfigurationError if used with block_type=moe.
#
# At seq_len=2048, fine_grained is safe and removes the attention score peak.
ACTIVATION_CHECKPOINTING = "fine_grained"

# ── Patch via yaml dict ────────────────────────────────────────────────────────
cfg = yaml.safe_load(CONFIG_PATH.read_text())

cfg["data"]["paths"]                 = list(TOKENIZED_DATA_PATHS)
cfg["model"]["moe_dropless"]         = False
cfg["model"]["moe_mlp_impl"]         = "sparse"
cfg["model"]["max_sequence_length"]  = 2048
cfg["precision"]                     = "amp_fp16"
cfg["activation_checkpointing"]      = ACTIVATION_CHECKPOINTING
cfg["global_train_batch_size"]       = GLOBAL_TRAIN_BATCH_SIZE
cfg["device_train_microbatch_size"]  = DEVICE_TRAIN_MICROBATCH_SIZE
cfg["save_folder"]                   = "/kaggle/working/runs/${run_name}"
cfg["compile"]                       = None

cfg["fsdp"].pop("use_orig_params", None)

out = yaml.dump(cfg, default_flow_style=False, sort_keys=False)
out = re.sub(r"'(\$\{[^}]+\}[^']*)'", r"\1", out)
CONFIG_PATH.write_text(out)

# ── Verify ────────────────────────────────────────────────────────────────────
check = yaml.safe_load(CONFIG_PATH.read_text())

errors = []
if check["data"]["paths"] != list(TOKENIZED_DATA_PATHS):
    errors.append(f"data.paths mismatch: {check['data']['paths']}")
if check["model"]["max_sequence_length"] != 2048:
    errors.append(f"max_sequence_length={check['model']['max_sequence_length']}")
if check["model"]["moe_dropless"] != False:
    errors.append("moe_dropless not False")
if check["activation_checkpointing"] != ACTIVATION_CHECKPOINTING:
    errors.append(f"activation_checkpointing={check['activation_checkpointing']}")
if check["global_train_batch_size"] != GLOBAL_TRAIN_BATCH_SIZE:
    errors.append(f"global_train_batch_size={check['global_train_batch_size']}")
if errors:
    raise RuntimeError("Config verification FAILED:\n  " + "\n  ".join(errors))

_overhead = {
    "fine_grained": "attn+norms only per block, skips MoE FFN (~20% compute overhead)",
    "null": "none (highest memory usage)",
}
print("=== Config OK ===")
print(f"  precision            : {check['precision']}")
print(f"  max_sequence_length  : {check['model']['max_sequence_length']}")
print(f"  moe_dropless         : {check['model']['moe_dropless']}")
print(f"  moe_mlp_impl         : {check['model']['moe_mlp_impl']}")
print(f"  activation_ckpt      : {check['activation_checkpointing']}  ({_overhead.get(str(check['activation_checkpointing']), '?')})")
print(f"  compile              : {check.get('compile')}")
print(f"  global_batch_size    : {check['global_train_batch_size']}")
print(f"  microbatch_size      : {check['device_train_microbatch_size']}")
print(f"  data.paths ({len(check['data']['paths'])} file(s)):")
for p in check["data"]["paths"]:
    print(f"    {p}")


In [ ]:
# ── Cell 8: Dry-run check (import + config validation, no GPU needed) ──────────
# Runs train.py directly (not via torchrun) to verify that all imports and
# config parsing succeed. The script will exit with a ValueError when it hits
# dist.init_process_group() — that's expected and treated as a pass, because
# it means every import and config check before that point succeeded.

env = os.environ.copy()
env["OLMO_TASK"]       = "model"
# OVERRIDE_DIR must come first so the fixed huggingface_hub shadows the stale
# system copy at /usr/local/lib/python3.12/dist-packages/.
env["PYTHONPATH"]      = f"{OVERRIDE_DIR}:{OLMOE_DIR}/OLMo:" + env.get("PYTHONPATH", "")
env["OMP_NUM_THREADS"] = "1"

result = subprocess.run(
    [sys.executable, f"{OLMOE_DIR}/OLMo/scripts/train.py", str(CONFIG_PATH)],
    capture_output=True, text=True, env=env, cwd=OLMOE_DIR
)

print("Dry run exit code:", result.returncode)
print("--- STDOUT ---")
print(result.stdout[:2000] or "(no stdout)")
print("--- STDERR (last 2000 chars) ---")
print(result.stderr[-2000:] or "(no stderr)")

# ── Classify the exit ─────────────────────────────────────────────────────────
has_import_error = any(
    marker in result.stderr
    for marker in ["ModuleNotFoundError", "ImportError", "cannot import name"]
)
has_config_error = any(
    marker in result.stderr
    for marker in ["OmegaConf", "yaml", "KeyError", "missing mandatory value",
                   "ConfigAttributeError", "MissingMandatoryValue"]
)
# Expected failure: script ran fine until dist.init_process_group() which
# requires RANK/WORLD_SIZE env vars only set by torchrun.
reached_distributed = "RANK expected, but not set" in result.stderr

if has_import_error:
    raise RuntimeError("Import error in dry run — fix dependencies before training.")
if has_config_error:
    raise RuntimeError("Config error in dry run — check olmoe-local.yml patches.")
if reached_distributed:
    print("\n✓ Dry run passed: all imports and config loaded OK.")
    print("  (Script stopped at dist.init_process_group — expected without torchrun.)")
elif result.returncode == 0:
    print("\n✓ Dry run passed cleanly.")
else:
    raise RuntimeError(
        f"Unexpected failure (exit {result.returncode}). See STDERR above."
    )


In [ ]:
# ── Cell 9: Launch training ────────────────────────────────────────────────────
import socket

# Find a free port to avoid 'address already in use' on notebook re-runs
def free_port():
    with socket.socket() as s:
        s.bind(('', 0))
        return s.getsockname()[1]

port = free_port()
print(f"Using rdzv port: {port}")
print(f"Launching on {N_GPU} GPU(s)...")

env = os.environ.copy()
env["OLMO_TASK"]       = "model"
# OVERRIDE_DIR must come first so the fixed huggingface_hub shadows the stale
# system copy at /usr/local/lib/python3.12/dist-packages/.
env["PYTHONPATH"]      = f"{OVERRIDE_DIR}:{OLMOE_DIR}/OLMo:" + env.get("PYTHONPATH", "")
env["OMP_NUM_THREADS"] = "4"
# Reduces CUDA memory fragmentation: allows the allocator to extend existing
# segments rather than failing when a contiguous block is unavailable.
# PyTorch recommends this explicitly in OOM error messages.
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# W&B vars already set in Cell 3 and propagated via os.environ

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    f"--nproc-per-node={N_GPU}",
    "--nnodes=1",
    "--node_rank=0",
    "--rdzv_backend=c10d",
    f"--rdzv_endpoint=localhost:{port}",
    "OLMo/scripts/train.py",
    str(CONFIG_PATH),
]

print("Command:", " ".join(cmd))
print("-" * 60)

# Stream stdout+stderr live so you see loss/step in real time
process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Training failed (exit code {process.returncode})")
print("\nTraining completed successfully.")
